### 第1步：普通的类属性问题

In [10]:
class Category:
    def __init__(self, name):
        self.name = name  # 直接存属性

# 使用
c = Category("科技")
print(c.name)  # 输出：科技
c.name = "互联网"
print(c.name)  # 输出：互联网

科技
互联网


问题来了：如果我想在设置name的时候自动做点事情（比如：检查不能为空、自动转大写、记录日志），怎么办？

### 第2步：用函数来模拟属性访问
一个笨办法：不直接用属性，而是用set和get函数

In [11]:
class Category:
    def __init__(self, name):
        self._name = name  # 用下划线表示"私有"

    def get_name(self):
        """获取名字"""
        print("有人获取名字了！")
        return self._name

    def set_name(self, value):
        """设置名字"""
        print(f"有人要把名字改成：{value}")
        if not value:  # 检查不能为空
            raise ValueError("名字不能为空")
        self._name = value

# 使用
c = Category("科技")
print(c.get_name())  # 输出：有人获取名字了！ 科技
c.set_name("互联网")  # 输出：有人要把名字改成：互联网
print(c.get_name())  # 输出：有人获取名字了！ 互联网

有人获取名字了！
科技
有人要把名字改成：互联网
有人获取名字了！
互联网


缺点：用起来太麻烦！每次都要写c.get_name()而不是c.name

### 第3步：Python的@property装饰器 - 让函数像属性一样用

In [12]:
class Category:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        """获取时自动调用这个函数"""
        print("有人获取名字了！")
        return self._name

    @name.setter
    def name(self, value):
        """设置时自动调用这个函数"""
        print(f"有人要把名字改成：{value}")
        if not value:
            raise ValueError("名字不能为空")
        self._name = value

# 使用 - 看起来就像在用普通属性！
c = Category("科技")
print(c.name)    # 输出：有人获取名字了！ 科技
c.name = "互联网" # 输出：有人要把名字改成：互联网
print(c.name)    # 输出：有人获取名字了！ 互联网

有人获取名字了！
科技
有人要把名字改成：互联网
有人获取名字了！
互联网


看到了吗？<br>
* c.name 看起来像属性，实际上调用了函数
* 这就是Python的属性拦截功能

#### 第4步：问题又来了 - 每个属性都要重复写@property
如果你有10个属性，就要写10次@property和@name.setter，太麻烦！

In [ ]:
class Category:
    def __init__(self, name, sort_order):
        self._name = name
        self._sort_order = sort_order

    # name的getter/setter
    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

    # sort_order的getter/setter - 又要重复写！
    @property
    def sort_order(self):
        return self._sort_order

    @sort_order.setter
    def sort_order(self, value):
        self._sort_order = value

#### 第5步：描述符就是"可重复使用的@property"

In [ ]:
# 这是一个描述符类 - 就像一个"属性模板"
class MyColumn:
    def __set_name__(self, owner, name):
        # 自动记录属性名（比如"name"或"sort_order"）
        self.private_name = f"_{name}"

    def __get__(self, instance, owner):
        # 当有人读取属性时，自动调用这里
        if instance is None:
            return self
        print(f"正在读取 {self.private_name} 的值")
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        # 当有人设置属性时，自动调用这里
        print(f"正在设置 {self.private_name} = {value}")
        setattr(instance, self.private_name, value)

# 使用描述符
class Category:
    name = MyColumn()        # 用模板创建name属性
    sort_order = MyColumn()  # 用同一个模板创建sort_order属性

    def __init__(self, name, sort_order):
        self.name = name          # 实际调用 MyColumn.__set__
        self.sort_order = sort_order

# 测试
c = Category("科技", 5)
print(c.name)        # 输出：正在读取 _name 的值   科技
print(c.sort_order)  # 输出：正在读取 _sort_order 的值   5
c.name = "互联网"     # 输出：正在设置 _name = 互联网

理解了吗？<br>
* MyColumn就是一个模板
* name = MyColumn() 创建了一个会"自动拦截"的属性
* 当你写c.name时，Python自动调用MyColumn.__get__
* 当你写c.name = xxx时，Python自动调用MyColumn.__set__

这是Python语言规范的一部分,Python官方文档明确规定了：<br>
* 当一个类中定义了__set_name__方法的描述符被赋值给类属性时
* Python解释器会自动调用这个__set_name__方法<br>
这不是巧合，而是Python解释器内部实现的机制。

#### 第6步：现在看SQLAlchemy的代码就懂了

In [ ]:
# SQLAlchemy的代码
#class Category(Base):
#    name: Mapped[str] = mapped_column(String(50))
#    sort_order: Mapped[int] = mapped_column(Integer, default=0)

# 本质就是：
# name = 一个描述符对象
# sort_order = 另一个描述符对象

c = Category()
c.name = "科技"  # 触发描述符的__set__
# ↓ 描述符内部做了：
#   1. 检查类型是不是字符串
#   2. 记录这个字段被修改了
#   3. 保存值到内部存储

print(c.name)    # 触发描述符的__get__
# ↓ 描述符内部做了：
#   1. 如果值还没从数据库加载，先去查数据库
#   2. 返回保存的值